# 🎨 Data Designer Tutorial: Structured Outputs and Jinja Expressions

#### 📚 What you'll learn

In this notebook, we will continue our exploration of Data Designer, demonstrating more advanced data generation using structured outputs and Jinja expressions.

If this is your first time using Data Designer, we recommend starting with the [first notebook](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/1-the-basics/) in this tutorial series.


# 🎨 Data Designer チュートリアル：構造化出力と Jinja 式

#### 📚 学習内容

このノートブックでは、Data Designer の学習を続け、構造化出力と Jinja 式を用いたより高度なデータ生成方法を解説します。

Data Designer を初めて使用する場合は、このチュートリアルシリーズの [最初のノートブック](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/1-the-basics/) から始めることをお勧めします。

### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.

- `DataDesigner` is the main interface for data generation.

### 📦 データデザイナーのインポート

- `data_designer.config` は設定APIへのアクセスを提供します。

- `DataDesigner` はデータ生成のためのメインインターフェースです。


In [1]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object that is used to interface with the library.

- When initialized without arguments, the [default model providers](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) are used.


### ⚙️ データデザイナーインターフェースの初期化

- `DataDesigner` は、ライブラリとのインターフェースに使用されるメインオブジェクトです。

- 引数なしで初期化した場合、[デフォルトのモデルプロバイダー](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) が使用されます。


In [3]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during the generation process.

- The "model alias" is used to reference the model in the Data Designer config (as we will see below).

- The "model provider" is the external service that hosts the model (see the [model config](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) docs for more details).

- By default, we use [build.nvidia.com](https://build.nvidia.com/models) as the model provider.


### 🎛️ モデル設定の定義

- 各 `ModelConfig` は、生成プロセスで使用できるモデルを定義します。

- 「モデルエイリアス」は、Data Designer の設定でモデルを参照するために使用されます（後述します）。

- 「モデルプロバイダ」は、モデルをホストする外部サービスです（詳細は [モデル設定](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) のドキュメントを参照してください）。

- デフォルトでは、モデルプロバイダとして [build.nvidia.com](https://build.nvidia.com/models) を使用します。


In [4]:
# This name is set in the model provider configuration.
MODEL_PROVIDER = "nvidia"

# The model ID is from build.nvidia.com.
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"

# We choose this alias to be descriptive for our use case.
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The Data Designer config defines the dataset schema and generation process.

- The config builder provides an intuitive interface for building this configuration.

- The list of model configs is provided to the builder at initialization.


### 🏗️ データデザイナー設定ビルダーの初期化

- データデザイナーの設定では、データセットのスキーマと生成プロセスを定義します。

- 設定ビルダーは、この設定を構築するための直感的なインターフェースを提供します。

- モデル設定のリストは、初期化時にビルダーに渡されます。


In [5]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

### 🧑‍🎨 Designing our data

- We will again create a product review dataset, but this time we will use structured outputs and Jinja expressions.

- Structured outputs let you specify the exact schema of the data you want to generate.

- Data Designer supports schemas specified using either json schema or Pydantic data models (recommended).

<br>

We'll define our structured outputs using [Pydantic](https://docs.pydantic.dev/latest/) data models

> 💡 **Why Pydantic?**
>
> - Pydantic models provide better IDE support and type validation.
>
> - They are more Pythonic than raw JSON schemas.
>
> - They integrate seamlessly with Data Designer's structured output system.


### 🧑‍🎨 データの設計

- 今回は、製品レビューデータセットを再度作成しますが、構造化出力とJinja式を使用します。

- 構造化出力を使用すると、生成するデータのスキーマを正確に指定できます。

- Data Designerは、JSONスキーマまたはPydanticデータモデル（推奨）で指定されたスキーマをサポートしています。

<br>

構造化出力は、[Pydantic](https://docs.pydantic.dev/latest/)データモデルを使用して定義します。

> 💡 **Pydanticを選ぶ理由**
>
> - Pydanticモデルは、IDEのサポートと型検証が優れています。

>
> - 生のJSONスキーマよりもPythonらしい記述が可能です。

>
> - Data Designerの構造化出力システムとシームレスに統合できます。


In [6]:
from decimal import Decimal
from typing import Literal

from pydantic import BaseModel, Field


# We define a Product schema so that the name, description, and price are generated
# in one go, with the types and constraints specified.
class Product(BaseModel):
    name: str = Field(description="The name of the product")
    description: str = Field(description="A description of the product")
    price: Decimal = Field(description="The price of the product", ge=10, le=1000, decimal_places=2)


class ProductReview(BaseModel):
    rating: int = Field(description="The rating of the product", ge=1, le=5)
    customer_mood: Literal["irritated", "mad", "happy", "neutral", "excited"] = Field(
        description="The mood of the customer"
    )
    review: str = Field(description="A review of the product")

Next, let's design our product review dataset using a few more tricks compared to the previous notebook.

次に、前回のノートブックよりも少し工夫を凝らして、製品レビューのデータセットを設計してみましょう。

In [7]:
# Since we often only want a few attributes from Person objects, we can
# set drop=True in the column config to drop the column from the final dataset.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="customer",
        sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
        params=dd.PersonFromFakerSamplerParams(),
        drop=True,
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Electronics",
                "Clothing",
                "Home & Kitchen",
                "Books",
                "Home Office",
            ],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_subcategory",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="product_category",
            values={
                "Electronics": [
                    "Smartphones",
                    "Laptops",
                    "Headphones",
                    "Cameras",
                    "Accessories",
                ],
                "Clothing": [
                    "Men's Clothing",
                    "Women's Clothing",
                    "Winter Coats",
                    "Activewear",
                    "Accessories",
                ],
                "Home & Kitchen": [
                    "Appliances",
                    "Cookware",
                    "Furniture",
                    "Decor",
                    "Organization",
                ],
                "Books": [
                    "Fiction",
                    "Non-Fiction",
                    "Self-Help",
                    "Textbooks",
                    "Classics",
                ],
                "Home Office": [
                    "Desks",
                    "Chairs",
                    "Storage",
                    "Office Supplies",
                    "Lighting",
                ],
            },
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="target_age_range",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=["18-25", "25-35", "35-50", "50-65", "65+"]),
    )
)

# Sampler columns support conditional params, which are used if the condition is met.
# In this example, we set the review style to rambling if the target age range is 18-25.
# Note conditional parameters are only supported for Sampler column types.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="review_style",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["rambling", "brief", "detailed", "structured with bullet points"],
            weights=[1, 2, 2, 1],
        ),
        conditional_params={
            "target_age_range == '18-25'": dd.CategorySamplerParams(values=["rambling"]),
        },
    )
)

# Optionally validate that the columns are configured correctly.
data_designer.validate(config_builder)

[10:54:41] [INFO] ✅ Validation passed


Next, we will use more advanced Jinja expressions to create new columns.

Jinja expressions let you:

- Access nested attributes: `{{ customer.first_name }}`

- Combine values: `{{ customer.first_name }} {{ customer.last_name }}`

- Use conditional logic: `{% if condition %}...{% endif %}`


次に、より高度な Jinja 式を使用して新しい列を作成します。

Jinja 式では、以下のことが可能です。

- ネストされた属性へのアクセス: `{{ customer.first_name }}`

- 値の結合: `{{ customer.first_name }} {{ customer.last_name }}`

- 条件付きロジックの使用: `{% if condition %}...{% endif %}`


In [8]:
# We can create new columns using Jinja expressions that reference
# existing columns, including attributes of nested objects.
config_builder.add_column(
    dd.ExpressionColumnConfig(name="customer_name", expr="{{ customer.first_name }} {{ customer.last_name }}")
)

config_builder.add_column(dd.ExpressionColumnConfig(name="customer_age", expr="{{ customer.age }}"))

config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="product",
        prompt=(
            "Create a product in the '{{ product_category }}' category, focusing on products  "
            "related to '{{ product_subcategory }}'. The target age range of the ideal customer is "
            "{{ target_age_range }} years old. The product should be priced between $10 and $1000."
        ),
        output_format=Product,
        model_alias=MODEL_ALIAS,
    )
)

# We can even use if/else logic in our Jinja expressions to create more complex prompt patterns.
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="customer_review",
        prompt=(
            "Your task is to write a review for the following product:\n\n"
            "Product Name: {{ product.name }}\n"
            "Product Description: {{ product.description }}\n"
            "Price: {{ product.price }}\n\n"
            "Imagine your name is {{ customer_name }} and you are from {{ customer.city }}, {{ customer.state }}. "
            "Write the review in a style that is '{{ review_style }}'."
            "{% if target_age_range == '18-25' %}"
            "Make sure the review is more informal and conversational.\n"
            "{% else %}"
            "Make sure the review is more formal and structured.\n"
            "{% endif %}"
            "The review field should contain only the review, no other text."
        ),
        output_format=ProductReview,
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[10:55:01] [INFO] ✅ Validation passed


### 🔁 Iteration is key – preview the dataset!

1. Use the `preview` method to generate a sample of records quickly.

2. Inspect the results for quality and format issues.

3. Adjust column configurations, prompts, or parameters as needed.

4. Re-run the preview until satisfied.


### 🔁 繰り返し検証が鍵です – データセットをプレビューしましょう！

1. `preview` メソッドを使用して、レコードのサンプルをすばやく生成します。

2. 結果の品質とフォーマットに問題がないか確認します。

3. 必要に応じて、列の設定、プロンプト、またはパラメータを調整します。

4. 満足できるまでプレビューを繰り返し実行します。


In [9]:
preview = data_designer.preview(config_builder, num_records=2)

[10:55:19] [INFO] 📸 Preview generation in progress
[10:55:19] [INFO]   |-- 🔒 Jinja rendering engine: secure
[10:55:19] [INFO] ✅ Validation passed
[10:55:19] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[10:55:19] [INFO] 🩺 Running health checks for models...
[10:55:19] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[10:55:19] [INFO]   |-- ✅ Passed!
[10:55:19] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue preview
[10:55:19] [INFO] 🗂️ llm-structured model config for column 'product'
[10:55:19] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[10:55:19] [INFO]   |-- model alias: 'nemotron-nano-v3'
[10:55:19] [INFO]   |-- model provider: 'nvidia'
[10:55:19] [INFO]   |-- inference parameters:
[10:55:19] [INFO]   |  |-- generation_type=chat-completion
[10:55:19] [INFO]   |  |-- max_parallel_requests=4
[10:55:19] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable

In [10]:
# Run this cell multiple times to cycle through the 2 preview records.
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                ┃ Value                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category    │ Home & Kitchen                                                                       │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ product_subcategory │ Decor                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ target_age_range    │ 50-65                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ review_style        │ brief                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ product             │ {                                                                                    │
│                     │     'name': 'Artisan Copper Lantern',                                                │
│                     │     'description': 'A hand‑crafted copper lantern featuring an elegant frosted glass │
│                     │ shade, designed to cast a warm, ambient glow. Includes a weathered patina that adds  │
│                     │ vintage charm to any indoor or covered outdoor space.',                              │
│                     │     'price': 79.99                                                                   │
│                     │ }                                                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer_review     │ {                                                                                    │
│                     │     'rating': 5,                                                                     │
│                     │     'customer_mood': 'happy',                                                        │
│                     │     'review': 'The Artisan Copper Lantern delivers a warm, ambient glow with its     │
│                     │ frosted glass shade and weathered patina, imparting an elegant vintage charm to both │
│                     │ indoor and covered outdoor settings. Its hand‑crafted copper construction reflects   │
│                     │ meticulous attention to detail, making it a distinguished lighting piece at a        │
│                     │ reasonable price.'                                                                   │
│                     │ }                                                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer_name       │ James Cox                                                                            │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer_age        │ 111                                                                                  │
└─────────────────────┴──────────────────────────────────────────────────────────────────────────────────────┘

In [11]:
# The preview dataset is available as a pandas DataFrame.
preview.dataset

,product_category,product_subcategory,target_age_range,review_style,customer_age,customer_name,product,customer_review
0,Home & Kitchen,Decor,50-65,brief,111,James Cox,"{'name': 'Artisan Copper Lantern', 'descriptio...","{'rating': 5, 'customer_mood': 'happy', 'revie..."
1,Books,Fiction,35-50,brief,88,David Hardy,{'name': 'Midlife Mystery Box: 12 Short Storie...,"{'rating': 4, 'customer_mood': 'happy', 'revie..."


### 📊 Analyze the generated data

- Data Designer automatically generates a basic statistical analysis of the generated data.

- This analysis is available via the `analysis` property of generation result objects.


### 📊 生成データの分析

- データデザイナーは、生成データの基本的な統計分析を自動的に生成します。

- この分析結果は、生成結果オブジェクトの `analysis` プロパティから利用できます。


In [12]:
# Print the analysis as a table.
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2                               │ 8                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category                 │           string │                         2 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ product_subcategory              │           string │                         2 (100.0%) │          subcategory │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ target_age_range                 │           string │                         2 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ review_style                     │           string │                          1 (50.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                             🗂️ LLM-Structured Columns                                              
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                       ┃               ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name           ┃     data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product               │          dict │                 2 (100.0%) │     264.5 +/- 0.5 │           63.5 +/- 2.1 │
├───────────────────────┼───────────────┼────────────────────────────┼───────────────────┼────────────────────────┤
│ customer_review       │          dict │                 2 (100.0%) │     316.5 +/- 1.5 │           80.5 +/- 2.1 │
└───────────────────────┴───────────────┴────────────────────────────┴───────────────────┴────────────────────────┘
                                                                                                                   
                                                        

### 🆙 Scale up!

- Happy with your preview data?

- Use the `create` method to submit larger Data Designer generation jobs.


### 🆙 スケールアップ！

- プレビューデータに満足いただけましたか？

- `create` メソッドを使用して、より大規模な Data Designer 生成ジョブを送信してください。


In [13]:
results = data_designer.create(config_builder, num_records=10, dataset_name="tutorial-2")

[10:56:12] [INFO] 🎨 Creating Data Designer dataset
[10:56:12] [INFO]   |-- 🔒 Jinja rendering engine: secure
[10:56:12] [INFO] ✅ Validation passed
[10:56:12] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[10:56:12] [INFO] 🩺 Running health checks for models...
[10:56:12] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[10:56:13] [INFO]   |-- ✅ Passed!
[10:56:13] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue builder
[10:56:13] [INFO] 🗂️ llm-structured model config for column 'product'
[10:56:13] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[10:56:13] [INFO]   |-- model alias: 'nemotron-nano-v3'
[10:56:13] [INFO]   |-- model provider: 'nvidia'
[10:56:13] [INFO]   |-- inference parameters:
[10:56:13] [INFO]   |  |-- generation_type=chat-completion
[10:56:13] [INFO]   |  |-- max_parallel_requests=4
[10:56:13] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable

In [14]:
# Load the generated dataset as a pandas DataFrame.
dataset = results.load_dataset()

dataset.head()

,product_category,product_subcategory,target_age_range,review_style,customer_age,customer_name,product,customer_review
0,Home & Kitchen,Furniture,50-65,brief,35,Tara Ellis,"{'description': 'A comfortable, supportive rec...","{'customer_mood': 'happy', 'rating': 4, 'revie..."
1,Electronics,Cameras,50-65,structured with bullet points,23,Thomas Wolfe,"{'description': 'A lightweight, user-friendly ...","{'customer_mood': 'happy', 'rating': 5, 'revie..."
2,Books,Fiction,18-25,rambling,109,Kelly Sanchez,{'description': 'A curated anthology of short ...,"{'customer_mood': 'happy', 'rating': 4, 'revie..."
3,Books,Textbooks,25-35,brief,28,Curtis Pierce,"{'description': 'A comprehensive, up-to-date t...","{'customer_mood': 'neutral', 'rating': 5, 'rev..."
4,Electronics,Headphones,50-65,detailed,32,Krystal Fowler,{'description': 'Premium over-ear headphones d...,"{'customer_mood': 'happy', 'rating': 5, 'revie..."


In [15]:
# Load the analysis results into memory.
analysis = results.load_analysis()

analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 8                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category                 │           string │                          5 (50.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ product_subcategory              │           string │                        10 (100.0%) │          subcategory │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ target_age_range                 │           string │                          5 (50.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ review_style                     │           string │                          4 (40.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                             🗂️ LLM-Structured Columns                                              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃                      ┃               ┃                            ┃       prompt tokens ┃     completion tokens ┃
┃ column name          ┃     data type ┃       number unique values ┃          per record ┃            per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ product              │          dict │                10 (100.0%) │       265.0 +/- 0.9 │         80.5 +/- 10.2 │
├──────────────────────┼───────────────┼────────────────────────────┼─────────────────────┼───────────────────────┤
│ customer_review      │          dict │                10 (100.0%) │      336.0 +/- 10.9 │        186.5 +/- 97.8 │
└──────────────────────┴───────────────┴────────────────────────────┴─────────────────────┴───────────────────────┘
                                                                                                                   
                                                        

## ⏭️ Next Steps

Check out the following notebook to learn more about:

- [Seeding synthetic data generation with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)

- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)



## ⏭️ 次のステップ

以下のノートブックを参照して、詳細を確認してください。

- [外部データセットを使用した合成データ生成のシード設定](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)

- [コンテキストとして画像を提供する](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [画像の生成](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)
